In [ ]:
import pandas as pd
df = pd.read_csv("healthcare_patient_journey.csv")

In [2]:
def interpret_patient(row):
    state = {
        "patient_id": row["patient_id"],
        "admission_type": row["admission_type"],
        "department": row["department"],
        "wait_time": row["wait_time_min"],
        "stay": row["length_of_stay_days"],
        "procedures": row["procedures_count"],
        "complications": row["complications"],
        "readmission": row["readmitted_30d"],
        "satisfaction": row["satisfaction_score"]
    }
    return state

In [3]:
def detect_bottlenecks(state):
    issues = []

    if state["wait_time"] > 30:
        issues.append("Admission delay")

    if state["stay"] > 5:
        issues.append("Extended hospitalization")

    if state["complications"] == 1:
        issues.append("Clinical complication")

    if state["readmission"] == 1:
        issues.append("Poor discharge coordination")

    return issues

In [4]:
def recommend_actions(issues):
    actions = []

    for issue in issues:
        if issue == "Admission delay":
            actions.append("Optimize triage scheduling")

        if issue == "Extended hospitalization":
            actions.append("Initiate discharge planning")

        if issue == "Clinical complication":
            actions.append("Escalate specialist review")

        if issue == "Poor discharge coordination":
            actions.append("Schedule post-discharge follow-up")

    return actions

In [5]:
def severity_score(state, issues):

    score = 0

    if state["wait_time"] > 30:
        score += 1

    if state["stay"] > 5:
        score += 1

    if state["complications"] == 1:
        score += 2

    if state["readmission"] == 1:
        score += 2

    if score >= 4:
        return "CRITICAL"
    elif score >= 2:
        return "HIGH"
    elif score == 1:
        return "MEDIUM"
    else:
        return "LOW"

In [6]:
def run_pipeline(row):
    state = interpret_patient(row)
    issues = detect_bottlenecks(state)
    actions = recommend_actions(issues)
    severity= severity_score(state, issues)

    return {
        "patient_id": state["patient_id"],
        "issues": issues,
        "actions": actions,
        "severity": severity 
    }

In [7]:
results = df.apply(run_pipeline, axis=1)
results = pd.DataFrame(results.tolist())

print(results.head())

   patient_id                                             issues  \
0           1  [Admission delay, Clinical complication, Poor ...   
1           2                                                 []   
2           3                                  [Admission delay]   
3           4                                                 []   
4           5     [Admission delay, Poor discharge coordination]   

                                             actions  severity  
0  [Optimize triage scheduling, Escalate speciali...  CRITICAL  
1                                                 []       LOW  
2                       [Optimize triage scheduling]    MEDIUM  
3                                                 []       LOW  
4  [Optimize triage scheduling, Schedule post-dis...      HIGH  


In [13]:
import json
import boto3

bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {"text": "Hello"}
            ]
        }
    ],
    "inferenceConfig": {
        "maxTokens": 200,
        "temperature": 0.5
    }
}

response = bedrock.invoke_model(
    modelId="amazon.nova-lite-v1:0",
    body=json.dumps(body),
    contentType="application/json"
)

result = json.loads(response["body"].read())
print(result)

{'output': {'message': {'content': [{'text': "Hello! How can I assist you today? If you have any questions or need information on a particular topic, feel free to ask. Whether it's about science, technology, history, or something else, I'm here to help."}], 'role': 'assistant'}}, 'stopReason': 'end_turn', 'usage': {'inputTokens': 1, 'outputTokens': 50, 'totalTokens': 51, 'cacheReadInputTokenCount': 0, 'cacheWriteInputTokenCount': 0}}


In [14]:
def nova_explain(patient_id, issues, actions, severity_score):

    prompt = f"""
You are a hospital care coordination AI.

Patient ID: {patient_id}

Issues:
{issues}

Recommended actions:
{actions}

Explain clearly why these issues occurred and why actions help.
Keep explanation concise and clinical.

Severity: {severity_score}
"""

    body = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"text": prompt}
                ]
            }
        ],
        "inferenceConfig": {
            "maxTokens": 200,
            "temperature": 0.3
        }
    }

    response = bedrock.invoke_model(
        modelId="amazon.nova-lite-v1:0",
        body=json.dumps(body),
        contentType="application/json"
    )

    result = json.loads(response["body"].read())

    return result["output"]["message"]["content"][0]["text"]

In [15]:
def run_pipeline_with_ai(row):
    state = interpret_patient(row)
    issues = detect_bottlenecks(state)
    actions = recommend_actions(issues)
    severity= severity_score(state, issues)
    explanation = nova_explain(state["patient_id"], issues, actions,severity)

    return {
        "patient_id": state["patient_id"],
        "issues": issues,
        "actions": actions,
        "severity": severity,
        "explanation": explanation
    }

In [16]:
sample_df = df.head(5)   # only 5 patients
results = sample_df.apply(run_pipeline_with_ai, axis=1)
results = pd.DataFrame(results.tolist())

In [17]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

In [18]:
results

,patient_id,issues,actions,severity,explanation
0,1,"[Admission delay, Clinical complication, Poor discharge coordination]","[Optimize triage scheduling, Escalate specialist review, Schedule post-discharge follow-up]",CRITICAL,"**Issues and Recommended Actions for Patient ID: 1**\n\n**Issues:**\n1. **Admission Delay**: Prolonged time from patient arrival to admission can result from inefficient triage processes or high patient volume.\n2. **Clinical Complication**: Complications can arise from delayed treatment, inadequate monitoring, or underlying disease severity.\n3. **Poor Discharge Coordination**: Ineffective discharge planning can lead to gaps in care, increased readmissions, and patient dissatisfaction.\n\n**Recommended Actions:**\n1. **Optimize Triage Scheduling**: Streamlining the triage process ensures timely assessment and prioritization of patients, reducing admission delays.\n2. **Escalate Specialist Review**: Prompt specialist intervention can address underlying issues more effectively, potentially preventing complications.\n3. **Schedule Post-Discharge Follow-Up**: Ensuring follow-up care helps manage recovery, monitor for complications, and address any gaps in care continuity.\n\n**Explanation:**\n- **Admission Delay**: Optimizing triage scheduling minimizes wait times, ensuring patients receive timely"
1,2,[],[],LOW,"Given the provided information, Patient ID 2 currently has no listed issues and no recommended actions. Here’s a concise explanation:\n\n**Explanation:**\n- **Issues:** The absence of issues indicates that the patient is currently stable and not presenting any acute or chronic conditions that require immediate attention or intervention.\n- **Recommended Actions:** Since there are no issues, no specific actions are necessary at this time. Continuous monitoring and regular follow-ups are standard practice to ensure the patient's health status remains stable.\n\n**Severity: LOW**\n- The low severity rating suggests that even if there were issues, they would be minor and manageable, not requiring urgent intervention. This rating supports the decision to maintain a routine care approach without immediate escalation."
2,3,[Admission delay],[Optimize triage scheduling],MEDIUM,"**Issue Explanation:**\n\nAdmission delays can occur due to several factors, including high patient influx, insufficient staffing, or inefficiencies in the triage process. These delays can lead to prolonged waiting times, increased patient discomfort, and potential exacerbation of medical conditions.\n\n**Recommended Actions:**\n\n**Optimize triage scheduling:**\n\n1. **Streamline Triage Process:** Implementing a more efficient triage system can help prioritize patients based on the urgency of their conditions, reducing delays.\n2. **Staff Allocation:** Ensure adequate staffing during peak hours to manage patient flow effectively.\n3. **Technology Utilization:** Use electronic health records (EHR) and predictive analytics to forecast patient volumes and allocate resources accordingly.\n4. **Staff Training:** Regular training for triage staff to enhance their efficiency and decision-making skills.\n\n**Why Actions Help:**\n\nOptimizing triage scheduling addresses the root causes of admission delays by ensuring that patients are assessed and admitted promptly. This leads to improved patient satisfaction, better clinical outcomes, and more efficient use of"
3,4,[],[],LOW,"**Patient ID: 4**\n\n**Issues:**\n- None identified at this time.\n\n**Recommended actions:**\n- None required currently.\n\n**Explanation:**\nThe patient currently has no identified issues, which is indicative of stable health conditions and effective management of any existing medical concerns. The absence of issues suggests that the patient's current treatment plan is effective, and there are no immediate concerns requiring intervention.\n\n**Severity: LOW**\n\nGiven the low severity, ongoing monitoring and routine follow-ups are recommended to ensure the patient'

In [19]:
results.to_csv("results.csv", index=False)